In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt-get update && sudo apt-get install -y build-essential

In [ ]:
!sudo apt-get install -y python3.10-dev

In [ ]:
%%capture
import os
import sys


!{sys.executable} -m pip install pip3-autoremove
!{sys.executable} -m pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install unsloth
!{sys.executable} -m pip install transformers==4.56.2
!{sys.executable} -m pip install --no-deps trl==0.22.2

In [ ]:
import sys
!{sys.executable} -m pip install nltk rouge_score absl-py
!{sys.executable} -m pip install bert_score
!{sys.executable} -m pip install scikit-learn
!pip install wandb


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA H100 80GB HBM3


In [3]:
from huggingface_hub import login

login()

In [4]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True,
    token=hf_token
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
from datasets import load_dataset
dataset = load_dataset("hungnm/vietnamese-medical-qa", split = "train")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9335 [00:00<?, ? examples/s]

In [7]:
data = dataset.train_test_split(test_size=0.1, shuffle=True, seed = 42)


In [8]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [9]:
train_data = data['train']
test_data = data['test']

In [10]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [11]:
train_data[0]

{'answer': 'Chào bạn. Với câu hỏi: “Tại sao mang thai 3 tháng đầu không thấy triệu chứng ốm nghén?” của bạn, bác sĩ xin trả lời như sau:\nNghén hay không nghén trong suốt thai kỳ phụ thuộc rất nhiều vào sự thay đổi hormone và sức khỏe của mỗi người. Vì thế, tình trạng nghén không phải là yếu tố quan trọng trong việc đánh giá sức khỏe của mẹ và thai nhi.\nTrường hợp của bạn, nếu mang thai 3 tháng đầu không ốm nghén mà vẫn cảm thấy khỏe mạnh, ăn uống ngon miệng, đây hoàn toàn là một dấu hiệu tốt. Nếu quá lo lắng, bạn hãy thường xuyên khám thai định kỳ theo lịch hẹn của bác sĩ để đảm bảo an toàn cho cả mẹ và bé nhé.\nCảm ơn bạn đã tin tưởng và gửi câu hỏi tới AI-Doctor. Chúc bạn có một thai kỳ khỏe mạnh. Trân trọng!',
 'question': 'Chào bác sĩ! Nhờ bác sĩ tư vấn giúp em tại sao mang thai 3 tháng đầu không thấy triệu chứng ốm nghén? Xin cảm ơn bác sĩ nhiều.'}

In [12]:
test_data[0]

{'answer': 'Chào bạn,\nHiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.\nBạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15-20 phút để cải thiện tình hình.\nNgoài ra nên tập thể dục vào sáng và tối các môn thể thao tăng cường vận động cơ xương khớp như chạy , đi bộ, tennis, cầu lông...\nNếu tình trạng kéo dài và không đỡ bạn nên sớm đi khám để bác sỹ có hướng điều trị cụ thể cho bạn.\nThân mến!',
 'question': "Xin chào bác sĩ,\n\nGần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.\nThông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.\n\nDo đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ"}

In [13]:
EOS_TOKEN = tokenizer.eos_token
def format_with_system_prompt(example):
  example['conversations'] = [
        {"role": "user", "content": example['question']},
        {"role": "assistant", "content": example['answer']}
  ]
  return example

train_data = train_data.map(format_with_system_prompt)
test_data = test_data.map(format_with_system_prompt)


def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

train_data = train_data.map(formatting_prompts_func, batched = True)
test_data = test_data.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/8401 [00:00<?, ? examples/s]

Map:   0%|          | 0/934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8401 [00:00<?, ? examples/s]

Map:   0%|          | 0/934 [00:00<?, ? examples/s]

In [14]:
train_data[0]

{'answer': 'Chào bạn. Với câu hỏi: “Tại sao mang thai 3 tháng đầu không thấy triệu chứng ốm nghén?” của bạn, bác sĩ xin trả lời như sau:\nNghén hay không nghén trong suốt thai kỳ phụ thuộc rất nhiều vào sự thay đổi hormone và sức khỏe của mỗi người. Vì thế, tình trạng nghén không phải là yếu tố quan trọng trong việc đánh giá sức khỏe của mẹ và thai nhi.\nTrường hợp của bạn, nếu mang thai 3 tháng đầu không ốm nghén mà vẫn cảm thấy khỏe mạnh, ăn uống ngon miệng, đây hoàn toàn là một dấu hiệu tốt. Nếu quá lo lắng, bạn hãy thường xuyên khám thai định kỳ theo lịch hẹn của bác sĩ để đảm bảo an toàn cho cả mẹ và bé nhé.\nCảm ơn bạn đã tin tưởng và gửi câu hỏi tới AI-Doctor. Chúc bạn có một thai kỳ khỏe mạnh. Trân trọng!',
 'question': 'Chào bác sĩ! Nhờ bác sĩ tư vấn giúp em tại sao mang thai 3 tháng đầu không thấy triệu chứng ốm nghén? Xin cảm ơn bác sĩ nhiều.',
 'conversations': [{'content': 'Chào bác sĩ! Nhờ bác sĩ tư vấn giúp em tại sao mang thai 3 tháng đầu không thấy triệu chứng ốm nghén

In [15]:
test_data[0]

{'answer': 'Chào bạn,\nHiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.\nBạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15-20 phút để cải thiện tình hình.\nNgoài ra nên tập thể dục vào sáng và tối các môn thể thao tăng cường vận động cơ xương khớp như chạy , đi bộ, tennis, cầu lông...\nNếu tình trạng kéo dài và không đỡ bạn nên sớm đi khám để bác sỹ có hướng điều trị cụ thể cho bạn.\nThân mến!',
 'question': "Xin chào bác sĩ,\n\nGần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.\nThông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.\n\nDo đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ",
 'conversations': [{'c

In [16]:
wandb login


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: Enter your choice:

  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


wandb: WARNING Invalid choice
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/admin/.netrc


In [17]:
import os
os.environ["WANDB_PROJECT"] = "Medical-Chatbot" 
os.environ["WANDB_LOG_MODEL"] = "false"             

In [18]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = test_data,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 4, # Set this for 1 full training run.
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/8401 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/934 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [19]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=64):   0%|          | 0/8401 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/8401 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/934 [00:00<?, ? examples/s]

Filter (num_proc=64):   0%|          | 0/934 [00:00<?, ? examples/s]

In [20]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                   Chào bạn\ntình trạng sốt ở trẻ nhỏ có liên quan đến rất nhiều nguyên nhan, nhiều mức độ khác nhau, nếu sốt kèm ho, chảy mũi, nôn, khó thở thì liên quan đến bệnh đường hô hấp. Nếu tre chỉ sốt đơn thuần, kéo dài cũng có thể do viêm dường tiết niệu mà mẹ không để ý đến, hoặc sốt cũng có thể do tình trạng nhiễm virus cấp tính thông thường 3-5 ngày sẽ đỡ, hoặc thậm chí là 1 biểu hiện của nhiễm khuẩn nặng. Mẹ có thể đưa bé di khám để xác định căn nguyên sốt, có hướng điều trị phù hợp nhất cho bé, tránh để tình trạng kéo dài hơn.\nThân mến<|im_end|>\n'

In [21]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,401 | Num Epochs = 4 | Total steps = 4,204
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
wandb: Currently logged in as: vietanhm6a (vietanhm6a-hanoi-university-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.092600
2,3.125800
3,2.970900
4,2.602700
5,2.117700
6,2.185300
7,1.947400
8,2.149000
9,1.869700
10,2.143800


train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█████
train/grad_norm,▁▂▁▂▁▂▂▃▂▂▃▃▂▃▃▃▃▃▃▅▃▄▄▄▄▅▄▄▅▅▇▇▅▆█▅▆▄▇▇
train/learning_rate,███▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
train/loss,▇▇▆▇▇▆█▆▅▇▆▆█▇▅▄▅▆▅▄▄▅▄▄▂▃▄▄▄▄▃▂▁▂▁▃▂▁▃▁
total_flos,2.545836282760366e+17
train/epoch,4
train/global_step,4204
train/grad_norm,3.90434
train/learning_rate,0.0
train/loss,0.6902


In [22]:
messages = [
    {"role" : "user", "content" : "Điều trị hội chứng mệt mỏi mạn tính như thế nào?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 4096, # Increase for longer outputs!
    # temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Hội chứng mệt mỏi mạn tính chưa có nguyên nhân được xác định, do đó chưa có cách điều trị đặc hiệu. Tuy nhiên, có thể điều trị bệnh bằng cách thay đổi lối sống và cách tiếp cận vấn đề.
Các biện pháp điều trị hội chứng mệt mỏi mạn tính:
Điều trị tâm lý: giúp người bệnh hiểu bệnh, chấp nhận bệnh và thích nghi với bệnh;
Điều chỉnh lối sống: thay đổi lối sống để thích nghi với bệnh. Nếu bệnh nhân làm việc quá nhiều sẽ làm nặng thêm bệnh, do đó cần giảm bớt công việc phù hợp với sức khỏe;
Sử dụng thuốc: sử dụng thuốc để nâng cao thể trạng, giảm triệu chứng, nâng cao chất lượng sống. Ví dụ: thuốc bổ máu nếu bị thiếu máu do bệnh lý này;
Chăm sóc bản thân: người bệnh cần tập thể dục nhẹ nhàng, ăn uống đầy đủ chất dinh dưỡng, nghỉ ngơi hợp lý, không nên thức khuya.<|im_end|>


In [23]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    token=hf_token
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [24]:
FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(base_model)


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Q

In [25]:
def infer_single(model, tokenizer, question):
    messages = [
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    output_ids = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=2048,
    )
    
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated.split("assistant")[-1].strip()


In [26]:
_ = infer_single(model, tokenizer, train_data[1]['question'])

In [27]:
print(_)

Chào bạn. TSH là gì? TSH là viết tắt của Thyroid Stimulating Hormone, hay còn gọi là Hormone kích thích tuyến giáp. Đây là một loại hormone tuyến yên, có tác dụng kích thích hoạt động bài tiết của tuyến giáp. Tuyến yên là bộ phận nằm ở dưới đáy hộp sọ, có vai trò như một trung tâm điều khiển hầu hết các bộ phận khác trong cơ thể. Khi não bộ nhận được tín hiệu rằng nồng độ hormone giáp trong máu thấp, nó sẽ phát ra tín hiệu cho tuyến yên hoạt động và tiết ra hormone kích thích tuyến giáp (TSH). Nếu nồng độ hormone giáp tiếp tục thấp, tuyến yên sẽ tăng sản xuất TSH. Ngược lại, nếu nồng độ hormone giáp cao, tuyến yên sẽ ngừng sản xuất TSH để tránh nồng độ hormone giáp quá cao. Thông qua đó, tuyến yên điều chỉnh nồng độ hormone tuyến giáp trong máu. Bác sĩ sẽ chỉ định xét nghiệm TSH khi bạn có các dấu hiệu và triệu chứng của suy giáp hoặc được bác sĩ nội tiết cảnh báo. Xét nghiệm TSH cũng được thực hiện khi bạn đang điều trị suy giáp hoặc muốn biết liệu liệu điều trị có hiệu quả hay không.

In [28]:
test_data = data['test']

In [29]:
test_data[0]

{'answer': 'Chào bạn,\nHiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.\nBạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15-20 phút để cải thiện tình hình.\nNgoài ra nên tập thể dục vào sáng và tối các môn thể thao tăng cường vận động cơ xương khớp như chạy , đi bộ, tennis, cầu lông...\nNếu tình trạng kéo dài và không đỡ bạn nên sớm đi khám để bác sỹ có hướng điều trị cụ thể cho bạn.\nThân mến!',
 'question': "Xin chào bác sĩ,\n\nGần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.\nThông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.\n\nDo đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ"}

In [31]:
model.push_to_hub("NgVietAnh41/Qwen3-4b-it-final-VietMedQA-qlora", token=hf_token)
tokenizer.push_to_hub("NgVietAnh41/Qwen3-4b-it-final-VietMedQA-qlora", token=hf_token)

README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-final-VietMedQA-qlora


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [30]:
test_data = data['test']
train_data = data['train']
test_data.to_json("test.jsonl", lines=True) 
train_data.to_json("train.jsonl", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

15183899

In [33]:
model.push_to_hub_gguf("NgVietAnh41/Qwen3-4b-it-final-VietMedQA-gguf", tokenizer, quantization_method = "q4_k_m", token = hf_token)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-final-VietMedQA-gguf
Unsloth: Cleaning up temporary files...


'NgVietAnh41/Qwen3-4b-it-final-VietMedQA-gguf'

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-test-VietMedQA-gguf
Unsloth: Cleaning up temporary files...


'NgVietAnh41/Qwen3-4b-it-test-VietMedQA-gguf'

In [32]:
model.push_to_hub_merged("NgVietAnh41/Qwen3-4b-it-final-VietMedQA", tokenizer, save_method = "merged_16bit", token = hf_token)


config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /home/admin/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [2:54:29<2:54:29, 10469.34s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [5:17:13<00:00, 9516.57s/it]   


Unsloth: Merge process complete. Saved to `/home/admin/NgVietAnh41/Qwen3-4b-it-final-VietMedQA`
